# 05 · Analysis: Entity–Role Binding under contradictory text
Interactive companion to `src/05_analysis.py` (which writes `reports/results.md` and all figures).
Definitions and decision rules: `PREREGISTRATION.md`. Set `MODELS = ["mock"]` to see the pipeline on the simulated model.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd() if (Path.cwd() / "analysis.py").exists() else Path.cwd() / "src"))
import pandas as pd, numpy as np
import analysis as A
from IPython.display import Image, display, Markdown
pd.set_option("display.width", 200, "display.max_columns", 30)

MODELS = None           # e.g. ["smolvlm2-2.2b", "internvl3-2b"]; None = every non-mock result
AUDITED_ONLY = False
items = A.load_items()
res = A.load_results(MODELS)
if AUDITED_ONLY:
    res = res[res.item_id.isin(items[items.audit_keep == True].item_id)]
res = res[res.item_id.isin(items.item_id)]
res.groupby(["model", "dataset"]).item_id.nunique()

## Competence set S_m and reassignment rates

In [ ]:
q4 = A.q4_table(res)
S = A.competence(res, q4)
display(S.groupby("model").size().rename("|S_m|"))
rr = A.rr_table(q4, S)
rr.pivot_table(index=["model", "dataset"], columns="condition", values="RR", observed=True).round(3)

## Capability panel and letter bias

In [ ]:
display(A.capability(res, q4).pivot_table(index=["model", "dataset"], columns="task", values="acc").round(3))
A.letter_bias(res)

## H1 · contradictory caption vs neutral / none (McNemar, cluster bootstrap)

In [ ]:
h1 = A.h1(q4, S, n_boot=2000)
h1[h1.test == "contra_2"].round(4)

## H2 · dose-response

In [ ]:
trend, gee = A.h2(q4, S, items)
display(trend)
for ds, g in gee.items():
    print(ds, {k: v for k, v in g.items() if k != "summary"})

## H3 / H4 · dissociation and awareness

In [ ]:
tax = A.taxonomy(res, q4, S)
display(A.h3_h4(tax).round(3))
tax.groupby(["model", "dataset"]).outcome.value_counts(normalize=True).unstack().round(3)

## Robustness and error analysis

In [ ]:
rob = A.robustness(res, S)
for k, v in rob.items():
    display(Markdown(f"**{k}**")); display(v.round(3) if hasattr(v, "round") else v)
A.difficulty_regression(q4, S, items)

## Figures (run `python src/05_analysis.py` first)

In [ ]:
for f in sorted((Path("..") if Path.cwd().name == "src" else Path(".")).joinpath("reports/figures").glob("fig*.png")):
    display(Markdown(f"**{f.name}**")); display(Image(filename=str(f)))